# Naive Persistence Baseline

## Description

The naive persistence forecast predicts the geomagnetic field $h$ hours from now as identical to its current value:

$$\hat{D}_{st}(t+h) = D_{st}(t)$$

No solar wind parameters, neutron monitor data, or learned relationships are used. Persistence serves as the **absolute lower bound** — any model that fails to outperform it at a given horizon provides no predictive value beyond the current observation.

Persistence performance degrades systematically with horizon. At h=1h the geomagnetic field changes little, making persistence a strong baseline. At h=21h the storm state may have evolved substantially, making it increasingly uninformative. The rate of this degradation defines the difficulty of the forecasting problem at each horizon and sets the reference level for all subsequent models.

All models are evaluated against this baseline via:
- **MASE** (Mean Absolute Scaled Error) — a model with MASE < 1 outperforms persistence; MASE > 1 means the model is operationally useless at that horizon regardless of absolute RMSE
- **Diebold-Mariano test** — statistical significance of the improvement over persistence

**Depends on:** `data/processed/feat_split.parquet`, `models/split_masks.pkl`, `models/context_constants.pkl`

## Mathematical Formulation

**Persistence forecast** at horizon $h$ for observation at time $t$:

$$\hat{D}_{st}(t+h) = D_{st}(t)$$

The forecast error is:

$$e_t^{(h)} = D_{st}(t+h) - \hat{D}_{st}(t+h) = D_{st}(t+h) - D_{st}(t)$$

This is simply the $h$-step difference of the $D_{st}$ series. The error grows with $h$ because the autocorrelation of $D_{st}$ decays — from PACF lag1 = 0.978 at h=1h toward zero at longer horizons.

**RMSE** at horizon $h$ over evaluation set $\mathcal{T}$:

$$\text{RMSE}^{(h)} = \sqrt{\frac{1}{|\mathcal{T}|} \sum_{t \in \mathcal{T}} \left( D_{st}(t+h) - D_{st}(t) \right)^2}$$

**Storm RMSE** — restricted to storm hours ($D_{st}(t+h) < -50$ nT):

$$\text{StormRMSE}^{(h)} = \sqrt{\frac{1}{|\mathcal{T}_s|} \sum_{t \in \mathcal{T}_s} \left( D_{st}(t+h) - D_{st}(t) \right)^2}$$

where $\mathcal{T}_s = \{t \in \mathcal{T} : D_{st}(t+h) < -50 \text{ nT}\}$.

**MASE** (Mean Absolute Scaled Error) [HYN06] — normalises MAE by the in-sample one-step persistence error on the training set:

$$\text{MASE}^{(h)} = \frac{\frac{1}{|\mathcal{T}|} \sum_{t \in \mathcal{T}} |e_t^{(h)}|}{\frac{1}{|\text{train}|-1} \sum_{t \in \text{train}} |D_{st}(t+1) - D_{st}(t)|}$$

The denominator is the mean absolute one-step difference on the training set — a scale-free normalisation that is independent of the forecast horizon. MASE = 1 means the model performs identically to one-step persistence on the training set. For persistence at horizon $h > 1$, MASE > 1 by construction — confirming it degrades with horizon.

**Diebold-Mariano test** [DM95] — tests whether the difference in forecast errors between two models is statistically significant:

$$DM = \frac{\bar{d}}{\sqrt{\hat{V}(\bar{d})}} \sim \mathcal{N}(0,1)$$

where $\bar{d} = \frac{1}{|\mathcal{T}|}\sum_{t \in \mathcal{T}} (e_{1,t}^2 - e_{2,t}^2)$ is the mean difference in squared errors between model 1 (persistence) and model 2 (candidate). A significant negative $DM$ statistic ($p < 0.05$) indicates that the candidate model produces smaller errors than persistence.

In [2]:
# ── Checkpoint Load ───────────────────────────────────────────────────────
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn
from sklearn.pipeline import Pipeline

from src.models import NaivePersistence
from src.evaluate import compute_metrics

# Load artifacts
feat  = pd.read_parquet('../data/processed/feat_split.parquet')
masks = joblib.load('../models/split_masks.pkl')
ctx   = joblib.load('../models/context_constants.pkl')

FEATURE_COLS      = ctx['FEATURE_COLS']
K_HORIZONS        = ctx['K_HORIZONS']
STORM_THR         = ctx['STORM_THR']

print(f'feat shape     : {feat.shape}')
print(f'K_HORIZONS     : {K_HORIZONS}')
print(f'STORM_THR      : {STORM_THR}')

ModuleNotFoundError: No module named 'src.models'